# Initial Raw Data Processing

Download UC Faculty zip and put "UC AWR_website_data with Home Dept 2013 -2024 (1).xlsx" into the colab to start


### Prerequisite

Download the data.zip folder from the google drive and extract it in the project directory.

The directory structure for this pipline should look like the following

```md
data/
    raw/
    processed/
data-pipline
    notebooks.ipynb
```

In [207]:
import os

# Check or create prerequisite folders
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../data/raw", exist_ok=True)
    

In [208]:
# Lossless data processing for the UC Faculty Dataset provided by a CPRA request to the UC System

import polars as pl

print("Scanning xlsx files...")


df_dict = pl.read_excel("../data/raw/UC AWR_website_data with Home Dept 2013 -2024 (1).xlsx",sheet_id=0)
df = pl.concat(df_dict.values())
df = df.rename(str.lower)

df = df.with_columns(
    pl.selectors.string().str.to_lowercase()
).with_columns(
    pl.selectors.string().str.strip_chars(),
).with_columns(
    # Since we just have year I figure datetime obj isnt worth it
    pl.col('year').cast(pl.Int16),

    # Change ints
    pl.col('base_pay').cast(pl.Int32),
    pl.col('overtime_pay').cast(pl.Int32),
    pl.col('other_pay').cast(pl.Int32),
    pl.col('gross_pay').cast(pl.Int32),


    # Cast to Categorical
    pl.col('campus').cast(pl.Categorical),
    pl.col('department').cast(pl.Categorical),
)


df.describe()

Scanning xlsx files...


statistic,year,campus,last_name,first_name,title_name,department,gross_pay,base_pay,overtime_pay,other_pay
str,f64,str,str,str,str,str,f64,f64,f64,f64
"""count""",3.660661e6,"""3660661""","""3660661""","""3660660""","""3660657""","""3660320""",3.660661e6,3.660661e6,3.660661e6,3.660661e6
"""null_count""",0.0,"""0""","""0""","""1""","""4""","""341""",0.0,0.0,0.0,0.0
"""mean""",2018.764224,null,null,null,null,null,56705.176119,48504.964208,847.00437,7353.209968
"""std""",3.458144,null,null,null,null,null,78820.283732,55530.470437,3942.074732,39005.584722
…,…,…,…,…,…,…,…,…,…,…
"""50%""",2019.0,null,null,null,null,null,34225.0,31786.0,0.0,0.0
"""75%""",2022.0,null,null,null,null,null,76860.0,71906.0,0.0,2500.0
"""max""",2024.0,null,"""zyzik""","""ângela""","""zone mech""",null,7.061667e6,1.965054e6,298933.0,6.761667e6


In [209]:
df.head()

year,campus,last_name,first_name,title_name,department,gross_pay,base_pay,overtime_pay,other_pay
i16,cat,str,str,str,cat,i32,i32,i32,i32
2024,"""asucla""","""abbassi""","""pouria""","""dir exec""","""executive director""",390489,356085,0,34405
2024,"""asucla""","""mehdian""","""kamran""","""dir""","""systems development""",247343,236450,0,10893
2024,"""asucla""","""baker""","""donna""","""dir""","""financial planning & analysis""",243072,232372,0,10700
2024,"""asucla""","""moyer""","""michelle""","""dir""","""human resources""",241336,231051,0,10285
2024,"""asucla""","""bolton""","""cynthia""","""dir""","""rest operations""",182084,174324,0,7760


### Load Data

In [210]:
import duckdb
duckdb.connect()


## Reimpliment Basic Data Filters

These were removed before but should have been adjusted to keep parttime faculty but still remove non-teaching faculty

In [211]:
# To try and only have educational faculty we fuzzy match on prof, lect, instr, and adj.

filter_str = """
SELECT *, CONCAT(title_name, department) as job_dep
FROM df
WHERE
    ( -- Basic Filters
    job_dep ILIKE '%prof%'
    OR job_dep ILIKE '%lect%'
    OR job_dep ILIKE '%adj%'
    OR job_dep ILIKE '%instr%'
    ) AND
    ( -- Advanced Filters mostly to exclude non-educational roles
    job_dep NOT ILIKE '%electrn%'
    AND job_dep NOT ILIKE '%profl%'
    AND job_dep NOT ILIKE '%maintenance%'
    AND job_dep NOT ILIKE '%collection%'
    AND job_dep NOT ILIKE '%budget%'
    AND job_dep NOT ILIKE '%instructional design%'
    AND job_dep NOT ILIKE '%elected%'
    AND job_dep NOT ILIKE '%adminstr%'
    AND job_dep NOT ILIKE '%services%'
    AND job_dep NOT ILIKE '%property%'
    AND job_dep NOT ILIKE '%electrocardiograph%'
    AND job_dep NOT ILIKE '%cntrl tchn%'
    ) AND
    -- Min Pay
    base_pay >= 0
"""

df = duckdb.sql(f"""
    {filter_str}
    """).pl()


duckdb.sql(f"""
CREATE OR REPLACE TABLE faculty_salaries AS
    SELECT *
    FROM df
    """)

### Role Column
No rows are dropped. `role` comes from `title_name` using the `UC_ROLES` rules in the next cell.
Table name for SQL statements is faculty_salaries

In [212]:
UC_ROLES = [
    ("recalled_emeriti", r"recall|emerit"),
    ("teaching_assistant", r"^teachg (asst|fellow)|^teaching assistant|^(special )?reader|tutor|^assoc in |^lang(uage)? asst"),
    ("grad_student_researcher", r"^gsr"),
    ("postdoc", r"postdoc"),
    ("undergrad_student_worker", r"^stdt"),
    ("medical_resident", r"^resid phys|^intern phys|^chief resid|resid-vet|pharmacy resid"),
    ("ladder_faculty", r"^(act |acting )?(asst |assoc )?prof-(ay|fy|hcomp)(?:(?:-b/e/e|-law|-1/9|-1/10))*$"
                       r"|^(sr )?lect p?soe|prof of teach(?:ing)?\b|teaching prof|^(asst |assoc )?professor of law$"
                       r"|^(asst |assoc )?(adjunct|adj\.?)\s+prof(?:essor)?\b|^asst prof(?:essor)? in\b"
                       r"|^(asst |assoc )?prof in res-(ay|fy)\b"
                       r"|^vstg (asst |assoc )?prof-(ay|fy)\b"
                       r"|^(asst |assoc )?prof-sft-vm\b|^assoc prof(?:essor)?\b|^(vis|vst|vstg|visiting) (asst |assoc )?prof(?:essor)?\b|^(asst |assoc )?prof(?:essor)? of\b"),
    ("lecturer", r"^(sr )?(lect|lecturer|lectr|lecr)\b|^adj instr?-(ay|fy)\b"),
    ("clinical_adjunct_faculty", r"\bprof\b|prof-|professor|clin instr|adj instr"),
    ("extension_instructor", r"^teacher-unex|^teacher-spec prog"),
    ("researcher_specialist_librarian", r"^(jr |asst |assoc )?specialist|proj(ect)? scientist|^(asst |assoc )?res-"
                                        r"|^(asst |assoc )?researcher|agron|coop(erative)? ext|librarian|^acad coord|^academic coord"),
]


def classify(title: pl.Expr, rules) -> pl.Expr:
    t = title.str.to_lowercase().str.strip_chars()
    expr = pl.lit("staff")
    for role, pattern in reversed(rules):
        expr = pl.when(t.str.contains(pattern)).then(pl.lit(role)).otherwise(expr)
    return expr


df = df.with_columns(classify(pl.col("title_name"), UC_ROLES).alias("role"))

duckdb.sql(f"""
CREATE OR REPLACE TABLE faculty_salaries AS
    SELECT *
    FROM df
    """)

Data info after adding roles

In [213]:
df.describe()

statistic,year,campus,last_name,first_name,title_name,department,gross_pay,base_pay,overtime_pay,other_pay,job_dep,role
str,f64,str,str,str,str,str,f64,f64,f64,f64,str,str
"""count""",399975.0,"""399975""","""399975""","""399975""","""399975""","""399954""",399975.0,399975.0,399975.0,399975.0,"""399975""","""399975"""
"""null_count""",0.0,"""0""","""0""","""0""","""0""","""21""",0.0,0.0,0.0,0.0,"""0""","""0"""
"""mean""",2018.743371,null,null,null,null,null,136765.827307,92233.723425,61.996172,44470.108539,null,null
"""std""",3.44169,null,null,null,null,null,154769.324267,83188.208458,1031.950817,99681.751481,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…
"""50%""",2019.0,null,null,null,null,null,98680.0,84942.0,0.0,4527.0,null,null
"""75%""",2022.0,null,null,null,null,null,201575.0,144092.0,0.0,44428.0,null,null
"""max""",2024.0,"""ucop""","""zyzik""","""zyrene""","""writer sr""","""young oak kim ctr dept""",3.974061e6,838117.0,94536.0,3.695451e6,"""writer srarts and lectures off…","""undergrad_student_worker"""


In [214]:
df.head()

year,campus,last_name,first_name,title_name,department,gross_pay,base_pay,overtime_pay,other_pay,job_dep,role
i16,str,str,str,str,str,i32,i32,i32,i32,str,str
2024,"""berkeley""","""malmendier""","""ulrike""","""prof-ay-b/e/e""","""haas core programs""",730483,548817,0,181667,"""prof-ay-b/e/ehaas core program…","""ladder_faculty"""
2024,"""berkeley""","""chatman""","""jennifer""","""prof-ay-b/e/e""","""haas core programs""",718700,445800,0,272900,"""prof-ay-b/e/ehaas core program…","""ladder_faculty"""
2024,"""berkeley""","""yaghi""","""omar""","""prof-ay""","""dept of chemistry""",690298,441667,0,248631,"""prof-aydept of chemistry""","""ladder_faculty"""
2024,"""berkeley""","""isacoff""","""ehud""","""prof-ay""","""molecular & cell biology""",680880,481058,0,199822,"""prof-aymolecular & cell biolog…","""ladder_faculty"""
2024,"""berkeley""","""farber""","""daniel""","""prof-ay-law""","""law""",677222,478333,0,198888,"""prof-ay-lawlaw""","""ladder_faculty"""


## Department Bucketing


### Amount of Existing Departments & Titles


In [215]:
duckdb.sql("""
SELECT COUNT(DISTINCT department) AS unique_departments
FROM faculty_salaries
           """).pl()

unique_departments
i64
2534


## Sample of the roles added for auditing

In [216]:
pl.Config.set_tbl_rows(20)

duckdb.sql("""
SELECT COUNT(department) AS count, department, role, title_name
FROM faculty_salaries
GROUP BY department, role, title_name
LIMIT 200
           """).pl()

count,department,role,title_name
i64,str,str,str
20,"""medctr-interns & residents""","""clinical_adjunct_faculty""","""hs asst clin prof-hcomp"""
157,"""medicine-endocrinology""","""clinical_adjunct_faculty""","""hs asst clin prof-hcomp"""
7,"""electrical engineering""","""staff""","""assoc dean"""
10,"""pathology department administr""","""clinical_adjunct_faculty""","""hs assoc clin prof-hcomp"""
206,"""ecology and evolutionary biolo""","""ladder_faculty""","""prof-ay"""
18,"""molecular & medical pharmacolo""","""clinical_adjunct_faculty""","""hs asst clin prof-hcomp"""
120,"""health policy and management""","""ladder_faculty""","""prof-ay"""
105,"""pediatrics-administration""","""clinical_adjunct_faculty""","""hs clin prof-hcomp"""
133,"""pathology department administr""","""ladder_faculty""","""prof-hcomp"""


### Checking the amount of departments in 

In [217]:

duckdb.sql("""
SELECT COUNT(*) AS count, department,
FROM faculty_salaries
GROUP BY department
ORDER BY count DESC
LIMIT 20
           """).pl()

count,department
i64,str
7652,"""elect & computer eng"""
6457,"""electrical engineering"""
6081,"""mathematics"""
5896,"""oid instructional improvement"""
5472,"""medicine"""
4408,"""electrical & computer engineer"""
4195,"""law"""
4004,"""history"""
3941,"""cultural & recreational affair"""


In [218]:
# Analyze Departments
duckdb.sql("""
SELECT DISTINCT department
FROM faculty_salaries
           """).pl()# .write_csv("Departments.csv")

department
str
"""medicine-gnrl int med-prm care"""
"""medicine-dept administration"""
"""electrical engineering"""
"""ecology and evolutionary biolo"""
"""molecular & medical pharmacolo"""
"""english"""
"""intellectual prop & indus spon"""
"""community health sciences"""
"""geography"""


### Creating The Proposed Buckets

* Engineering
* Computer Science
* Science & Mathematics
* Medicine & Heath Sciences
* Business, Economics & Finance
* Agriculture, Food & Animal
* Architecture & Environmental Design
* Humanities & Languages
* Social Sciences & Communication
* Education & Public Administration

In [219]:
# Removing the staff from the schools that are explicitly Hospital staff

duckdb.sql("""
CREATE OR REPLACE TABLE faculty_salaries AS
SELECT *
FROM faculty_salaries
WHERE NOT (
    -- UC "Hospital" campuses
    COALESCE(campus, '') IN (
        'davis', 'irvine', 'los angeles',
        'san diego', 'san francisco', 'riverside'
    )
    AND (
        (
            COALESCE(title_name, '') ILIKE '%clin%'
            AND COALESCE(title_name, '') NOT ILIKE '%law%'
        )
        OR COALESCE(title_name, '') ILIKE '%nurse%'
        OR COALESCE(title_name, '') ILIKE '%physcn%'
        OR COALESCE(title_name, '') ILIKE '%physician%'
        OR COALESCE(title_name, '') ILIKE 'resid phys%'
        OR COALESCE(title_name, '') ILIKE 'intern phys%'
        OR COALESCE(title_name, '') ILIKE 'phys ther%'
    )
)
""")

In [220]:
bucket_sql = """
SELECT *,
CASE
    WHEN
        department ILIKE '%computer%'
        OR department ILIKE '%comp sci%'
        OR department ILIKE '%cmptr sci%'
        OR department ILIKE '%computing%'
        OR department ILIKE '%software%'
        OR department ILIKE '%informatics%'
        OR department ILIKE '%cyber%'
        OR department ILIKE '%data science%'
        OR department ILIKE '%information studies%'
        OR department ILIKE '%school of info%'
    THEN 'Computer Science'

--- Adding Catches Before we filter out all the Hospital People 
    WHEN
        department ILIKE '%bioengineering%'
    THEN 'Engineering'

    WHEN
        department ILIKE '%biochemistry and biophysics%'
        OR department ILIKE '%biochem & molecular med%'
        OR department ILIKE '%microbiology and immunology%'
        OR department ILIKE '%medical microbiology & imm%'
        OR department ILIKE '%cell and tissue biology%'
        OR department ILIKE '%cell biology & human anat%'
        OR department ILIKE '%anat physio & cell biology%'
        OR department ILIKE '%physiology & membrane biol%'
        OR department ILIKE '%plant pathology%'
        OR department ILIKE '%pharmaceutical chemistry%'
        OR department ILIKE '%pharmaceutical science%'
        OR department ILIKE '%cellular & molecular medicine%'
        OR department ILIKE '%cellular & molecular pharmacol%'
        OR department ILIKE '%molecular & medical pharmacol%'
        OR department ILIKE '%molecular bio sciences%'
        OR department ILIKE 'pharmacology'
        OR department ILIKE 'med:%pharmacology'
        OR department ILIKE 'm!_anatomy%' ESCAPE '!'
        OR department ILIKE 'm!_physiology' ESCAPE '!'
        OR department ILIKE 'm!_cmp (cellular molecular pha)' ESCAPE '!'
        OR department ILIKE 'som neuroscience'
        OR department ILIKE 'chemical sciences'
        OR department ILIKE 'chemistry & chemical biology'
    THEN 'Science & Mathematics'    

    WHEN
        department ILIKE '%medicine%'
        OR department ILIKE '%surgery%'
        OR department ILIKE '%vm:%'
        OR department ILIKE '%med:%'
        OR (
            campus = 'san francisco'
            AND (
                department ILIKE 'm!_%' ESCAPE '!'
                OR department ILIKE 'n!_%' ESCAPE '!'
                OR department ILIKE 'd!_%' ESCAPE '!'
                OR department ILIKE 'p!_%' ESCAPE '!'
                OR department ILIKE 'h!_%' ESCAPE '!'
            )
        )
        OR department ILIKE '%nurs%'
        OR department ILIKE '%dentistry%'
        OR department ILIKE '%pharma%'
        OR department ILIKE '%gynecology%'
        OR department ILIKE '%urology%'
        OR department ILIKE '%oncology%'
        OR department ILIKE '%family med%'
        OR department ILIKE '%radiation%'
        OR department ILIKE '%ophthalmology%'
        OR department ILIKE '%pathology%'
        OR department ILIKE '%peds%'
        OR department ILIKE '%dermatology%'
        OR department ILIKE '%pediatrics%'
        OR department ILIKE '%anesthesiology%'
        OR department ILIKE '%optometry%'
        OR department ILIKE '%psychiatry%'
        OR department ILIKE '%emerg med%'
        OR department ILIKE '%radiolog%'
        OR department ILIKE '%radiation%'
        OR department ILIKE '%anesthes%'
        OR department ILIKE '%public health%'
        OR department ILIKE '%med ed administration%'
        OR department ILIKE '%deans office-school of medicin%'
        OR department ILIKE '%semel institute%'
        OR department ILIKE '%school of med%'
        OR department ILIKE '%dom cardiology%'
        OR department ILIKE '%ophth clinical%'
        OR department ILIKE '%med-campus%'
        OR department ILIKE '%dntl sci%'
        OR department ILIKE '%ob/gyn%'
        OR department ILIKE '%ambulatory%'
        OR department ILIKE '%electronic health record%'
        OR department ILIKE '%electrocardiology%'
        OR department ILIKE '%epidemiology%'
        OR department ILIKE '%otolaryngology%'
        OR department ILIKE '%hematology%'
        OR department ILIKE '%school of public health%'
        OR department ILIKE '%health policy and management%'
        OR department ILIKE '%health sciences%'
        OR department ILIKE '%lppi: instr & research%'
        OR department ILIKE 'dom %'
        OR department ILIKE 'som %'
        OR department ILIKE 'som-%'
        OR department ILIKE '%obgyn%'
        OR department ILIKE '%sfgh%'
        OR department ILIKE 'psych-%'
        OR department ILIKE 'fcm-%'
        OR department ILIKE '%health science%'
        OR department ILIKE '%health care%'
        OR department ILIKE '%health serv%'
        OR department ILIKE '%pub health%'
        OR department ILIKE '%applied health%'
        OR department ILIKE '%kinesiology%'
        OR department ILIKE '%occupational therapy%'
        OR department ILIKE '%audiology%'
        OR department ILIKE '%clinical laboratory%'
        OR department ILIKE '%clinical science%'
        OR department ILIKE '%gerontology%'
        OR department ILIKE '%marital and family therapy%'
        OR department ILIKE '%neurology%'
        OR department ILIKE '%physical therapy%'
        OR department ILIKE '%physical med%'
        OR department ILIKE '%hospital%'
        OR department ILIKE '%electrophysiology%'
        OR department ILIKE '%physician%'
        OR department ILIKE '%medical ed%'
        OR department ILIKE '%dental%'
    THEN NULL

   WHEN
        department ILIKE 'phys ed program'
        OR department ILIKE 'physical education program'
    THEN 'Education'

    WHEN
        department ILIKE 'physical activities%'
    THEN 'Other'

    WHEN
        department ILIKE '%business%'
        OR department ILIKE '%econom%'
        OR department ILIKE 'econ%'
        OR department ILIKE '% econ%'
        OR department ILIKE '%finance%'
        OR department ILIKE '%of managemen%'
        OR department ILIKE '%haas core%'
        OR department ILIKE '%technology management program%'
        OR department ILIKE '%grad.sch. int.relat./pac.stud.%'
        OR (
            department ILIKE '%accounting%'
            AND department NOT ILIKE '%office%' -- dodges some non educators
        )
        OR department ILIKE '%accountancy%'
        OR department ILIKE '%marketing%'
        OR department ILIKE 'management'
        OR department ILIKE '%management and human resources%'
        OR department ILIKE '%management & human resources%'
    THEN 'Business, Economics & Finance'

    WHEN
        department ILIKE '%engineer%'
        OR department ILIKE '%engr%'
        OR department ILIKE '%mech%'
        OR department ILIKE '%civil%'
        OR department ILIKE '%areo%'
        OR department ILIKE '%electric%'
        OR department ILIKE '%soe-ams%'
        OR department ILIKE '%chemical%'
        OR department ILIKE '%industrial%'
        OR department ILIKE '%material%'
        OR department ILIKE '%soece-ee-tm-cm%'
    THEN 'Engineering'

    WHEN
        department ILIKE '%math%'
        OR department ILIKE '%stat%'
        OR department ILIKE '%statistics%'
        OR department ILIKE '%bio%'
        OR department ILIKE '%neuro%'
        OR department ILIKE '%phys%'
        OR department ILIKE '%land air & water resources%'
        OR department ILIKE '%climate/atmos sci/phy oceanog%'
        OR department ILIKE '%ocean sciences%'
        OR department ILIKE '%entomology%'
        OR department ILIKE '%physics%'
        OR department ILIKE '%astro%'
        OR department ILIKE '%chem%'
        OR department ILIKE '%sio dep%'
        OR department ILIKE '%molecular, cell%'
        OR department ILIKE '%earth%'
        OR department ILIKE '%oceanic%'
        OR department ILIKE '%ecology%'
        OR department ILIKE '%genetics%'
        OR department ILIKE '%igpp%'
        OR department ILIKE '%geosciences%'
        OR department ILIKE '%oceanography%'
        OR department ILIKE '%geology%'
        OR department ILIKE '%viticulture%'
        OR department ILIKE '%nematology%'
        OR department ILIKE '%demography%'
    THEN 'Science & Mathematics'

    WHEN
        department ILIKE '%plant science%'
        OR department ILIKE '%agriculture%'
        OR department ILIKE '%espm%'
        OR department ILIKE '%nutrition%'
        OR department ILIKE '%inst of environment and sustai%'
        OR department ILIKE '%energy & resources group%'
        OR department ILIKE '%animal%'
        OR department ILIKE '%food science%'
        OR department ILIKE '%food safety%'
    THEN 'Agriculture, Food & Animal'

    WHEN
        department ILIKE '%architec%'
        OR department ILIKE '%planning%'
        OR department ILIKE '%urban%'
        OR department ILIKE '%landscape arch%'
        OR department ILIKE '%environmental%'
    THEN 'Architecture & Environmental Design'

    WHEN
        department ILIKE 'law'
        OR department ILIKE 'law/%'
        OR department ILIKE 'law %'
        OR department ILIKE '%school of law%'
        OR department ILIKE '%law school%'
        OR department ILIKE '% law %'
        OR department ILIKE '% law'
        OR department ILIKE '%legal%'
        OR department ILIKE 'jd program'
    THEN 'Public Administration & Law'

    WHEN
        department ILIKE '%education%'
        OR department ILIKE '%educational%'
        OR department ILIKE '%teacher%'
        OR department ILIKE '%teaching%'
        OR department ILIKE '%edleadership%'
        OR department ILIKE '%ed doctorate%'
        OR department ILIKE '%undergrad ed%'
        OR department ILIKE '%extended ed%'
        OR department ILIKE '%learning design%'
        OR department ILIKE '%instructional technology%'
        OR department ILIKE '%lifelong learning%'
    THEN 'Education'

    WHEN
        department ILIKE '%public administration%'
        OR department ILIKE '%public policy%'
        OR department ILIKE '%public affairs%'
        OR department ILIKE '%pub pol%'
        OR department ILIKE '%policy&strategy%'
        OR department ILIKE '%homeland security%'
    THEN 'Public Administration & Law'

    WHEN
        department ILIKE '%language%'
        OR department ILIKE '%humanities%'
        OR department ILIKE '%english%'
        OR department ILIKE '%literature%'
        OR department ILIKE '%writing%'
        OR department ILIKE '%history%'
        OR department ILIKE '%philosophy%'
        OR department ILIKE '%french%'
        OR department ILIKE '%spanish%'
        OR department ILIKE '%german%'
        OR department ILIKE '%japanese%'
        OR department ILIKE '%dance%'
        OR department ILIKE '%music%'
        OR department ILIKE '%division of arts%'
        OR department ILIKE '%classics%'
        OR department ILIKE '%film%'
        OR department ILIKE '%visual arts%'
        OR department ILIKE '%theatre%'
        OR department ILIKE '%religious%'
        OR department ILIKE '%theater%'
        OR department ILIKE '%asian lang%'
        OR department ILIKE '%eastern lang%'
        OR department ILIKE '%world arts%'
        OR department ILIKE '%southeast asian%'
        OR department ILIKE '%drama department%'
        OR department ILIKE '%studio art%'
        OR department ILIKE 'art'
        OR department ILIKE 'arts'
        OR department ILIKE 'art %'
        OR department ILIKE 'arts %'
        OR department ILIKE 'arts-%'
        OR department ILIKE '% art'
        OR department ILIKE '% arts'
        OR department ILIKE '% art %'
        OR department ILIKE '% arts %'
        OR department ILIKE '%design program%'
        OR department ILIKE '%rhetoric%'
        OR department ILIKE '%near eastern%'
        OR department ILIKE '%performance, play & design%'
        OR department ILIKE '%drama%'
        OR department ILIKE '%hispanic%'
        OR department ILIKE '%italian%'
        OR department ILIKE '%scandinavian%'
        OR department ILIKE '%slavic%'
        OR department ILIKE '%greek & roman%'
        OR department ILIKE '%department of design%'
        OR department ILIKE '%media%'
        OR department ILIKE '%institute for society and gene%'
        OR department ILIKE '%college of creative studies%'    
        OR department ILIKE '%journalism%'
    THEN 'Humanities & Languages'

    WHEN
        department ILIKE '%anthropology%'
        OR department ILIKE '%linguistics%'
        OR department ILIKE '%department of chicana%'
        OR department ILIKE '%asian-american%'
        OR department ILIKE '%asian%'
        OR department ILIKE '%chicano%'
        OR department ILIKE '%american studies%'
        OR department ILIKE '%italian studies%'
        OR department ILIKE '%native american studies%'
        OR department ILIKE '%african%'
        OR department ILIKE '%Social Science%'
        OR department ILIKE '%sociology%'
        OR department ILIKE '%political science%'
        OR department ILIKE '%politics%'
        OR department ILIKE '%ethnic stud%'
        OR department ILIKE '%women%'
        OR department ILIKE '%psychology%'
        OR department ILIKE '%psychological%'
        OR department ILIKE '%cognitive science%'
        OR department ILIKE '%global & intl studies%'
        OR department ILIKE '%soc sci undergrad%'
        OR department ILIKE '%comm%'
        OR department ILIKE '%gender%'
        OR department ILIKE '%child%'
        OR department ILIKE '%geography%'
        OR department ILIKE '%human ecology%'
        OR department ILIKE '%social welfare%'
        OR department ILIKE '%global & international studies%'
        OR department ILIKE '%african am studies%'
        OR department ILIKE '%black studies%'
        OR department ILIKE '%studies%'
        OR department ILIKE '%science & technology studies%'    
        OR department ILIKE 'm!_anthro%' ESCAPE '!'   
    THEN 'Social Sciences & Communication'
    
    ELSE 'Other'
END AS dep_bucket
FROM faculty_salaries
WHERE dep_bucket IS NOT NULL
"""

duckdb.sql(f"""
CREATE OR REPLACE TABLE faculty_salaries AS
{bucket_sql}
""")

In [221]:
# Send UnBucketable Departments to Other
duckdb.sql("""
UPDATE faculty_salaries
SET dep_bucket = 'Other'
WHERE department IN (
    'continuing & professional ed.',
    'faculty',
    'intellectual prop & indus spon',
    'edu/instruction & research',
    'university - davis',
    'elect eng div operations',
    'mae department accounts',
    'administration',
    'merage/instruction & academic',
    'the preuss school',
    'public health administration',
    'instructional development',
    'geffen academy at ucla',
    'gspp department ops',
    'dean-school of env sci & mgmt',
    'continuing & professional ed. ',
    'continuing and professional ed',
    'structural system facility',
    'chancellor ucsf',
    'oid instructional improvement',
    'tep',
    'sph administration            ',
    'information technology svcs',
    'ofc of instructional developme',
    'geffen academy at ucla        ',
    '',
)
""")

# How many departments are accounted for in each bucket


In [222]:
pl.Config.set_tbl_rows(11)

# How many departments are accounted for in each bucket
duckdb.sql("""
SELECT COUNT(DISTINCT department) AS count, dep_bucket
FROM faculty_salaries
GROUP BY dep_bucket
ORDER BY count DESC
           """).pl()

count,dep_bucket
i64,str
818,"""Other"""
210,"""Humanities & Languages"""
206,"""Science & Mathematics"""
145,"""Social Sciences & Communicatio…"
105,"""Engineering"""
37,"""Education"""
32,"""Business, Economics & Finance"""
29,"""Computer Science"""
26,"""Architecture & Environmental D…"


### Check what department are in the Other bucket

In [223]:
pl.Config.set_tbl_rows(7)
duckdb.sql("""
SELECT COUNT(*) as count, department
FROM faculty_salaries
WHERE dep_bucket = 'Other'
GROUP BY department
HAVING count > 1
ORDER BY count DESC
           """).pl()

count,department
i64,str
5896,"""oid instructional improvement"""
3941,"""cultural & recreational affair"""
3784,"""intercollegiate athletics"""
3554,"""vcsa campus recreation"""
…,…
2,"""electron microscope facility"""
2,"""essentials"""
2,"""berkeley seismological lab"""


In [224]:
df = duckdb.sql("""
    SELECT *, CONCAT(first_name, ' ', last_name) AS name
    FROM faculty_salaries
""").pl()

df = df.drop(["first_name", "last_name", "job_dep"])

df = df.rename({
    "department": "original_department",
    "dep_bucket": "department",
    "title_name": "orig_title",
})

Condense title to tenure_track or lecturer based on ladder_faculty and lecturer UC_ROLE designations

In [225]:
# Preserve the detailed title if it hasn't already been renamed.
if "title" in df.columns and "orig_title" not in df.columns:
    df = df.rename({"title": "orig_title"})

# Classify using the existing UC_ROLES assignments.
df = df.with_columns(
    pl.when(pl.col("role") == "ladder_faculty")
    .then(pl.lit("tenure_track"))
    .when(pl.col("role") == "lecturer")
    .then(pl.lit("lecturer"))
    .otherwise(pl.lit(None, dtype=pl.String))
    .alias("final_title")
)

In [226]:
# Overall matching counts
display(
    df.select(
        pl.len().alias("total_rows"),
        pl.col("final_title").is_not_null().sum().alias("matched_rows"),
        pl.col("final_title").is_null().sum().alias("unmatched_rows"),
    )
)

# Counts by classification
display(
    df.group_by("final_title")
    .len(name="rows")
    .sort("rows", descending=True)
)

# All unmatched titles, most frequent first
unmatched_titles = (
    df.filter(pl.col("final_title").is_null())
    .group_by("orig_title")
    .len(name="rows")
    .sort("rows", descending=True)
)

print("Unmatched titles:")
with pl.Config(tbl_rows=-1, fmt_str_lengths=150):
    display(unmatched_titles)

# Matched examples for manual validation
for category in ["tenure_track", "lecturer"]:
    matches = df.filter(pl.col("final_title") == category)
    print(f"\nExamples: {category}")
    display(
        matches.select("name", "orig_title", "role", "final_title")
        .sample(n=min(5, matches.height), seed=42)
    )

total_rows,matched_rows,unmatched_rows
u32,u32,u32
270497,194448,76049


final_title,rows
str,u32
"""tenure_track""",122667
null,76049
"""lecturer""",71781


Unmatched titles:


orig_title,rows
str,u32
"""recr prg instr 2""",7913
"""recreation prg instr""",6945
"""recr prg instr""",6866
"""teachg asst-gship""",5301
"""stdt 1 non uc""",5029
"""teacher-unex-contract yr""",4258
"""gsr-partial fee rem""",3450
"""stdt 1""",2367
"""gsr-tuit & fee rem""",2070



Examples: tenure_track


name,orig_title,role,final_title
str,str,str,str
"""stephen atkins greene""","""adj prof-ay-b/e/e""","""ladder_faculty""","""tenure_track"""
"""jeffrey knapp""","""prof-ay""","""ladder_faculty""","""tenure_track"""
"""james charles nieh""","""prof-ay""","""ladder_faculty""","""tenure_track"""
"""alan miller""","""vis assoc prof""","""ladder_faculty""","""tenure_track"""
"""sreya kolay""","""asst prof-ay-b/e/e""","""ladder_faculty""","""tenure_track"""



Examples: lecturer


name,orig_title,role,final_title
str,str,str,str
"""nandita jha""","""lect in summer session""","""lecturer""","""lecturer"""
"""olga azarenko""","""lect-ay-1/9""","""lecturer""","""lecturer"""
"""steven e. larson""","""lect-miscellaneous/part time""","""lecturer""","""lecturer"""
"""theodore a shugar""","""lect-ay-1/9-continuing""","""lecturer""","""lecturer"""
"""soraya sablo-sutton""","""lect-ay""","""lecturer""","""lecturer"""


In [227]:
rows_before = df.height
df = df.drop_nulls(subset=["final_title"])

print(f"Dropped {rows_before - df.height:,} rows")
print(f"Remaining: {df.height:,} rows")

Dropped 76,049 rows
Remaining: 194,448 rows


In [228]:
df.write_parquet("../data/processed/golden_uc_2013-2024.parquet")
df.describe()

statistic,year,campus,orig_title,original_department,gross_pay,base_pay,overtime_pay,other_pay,role,department,name,final_title
str,f64,str,str,str,f64,f64,f64,f64,str,str,str,str
"""count""",194448.0,"""194448""","""194448""","""194431""",194448.0,194448.0,194448.0,194448.0,"""194448""","""194448""","""194448""","""194448"""
"""null_count""",0.0,"""0""","""0""","""17""",0.0,0.0,0.0,0.0,"""0""","""0""","""0""","""0"""
"""mean""",2018.734366,null,null,null,117935.753621,99283.908063,0.559913,18651.286112,null,null,null,null
"""std""",3.418068,null,null,null,103074.596538,82436.192204,73.009408,32441.49468,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…
"""50%""",2019.0,null,null,null,104078.0,93750.0,0.0,3025.0,null,null,null,null
"""75%""",2022.0,null,null,null,174600.0,147733.0,0.0,25300.0,null,null,null,null
"""max""",2024.0,"""ucop""","""vstg prof-ay-1/9-bus/econ/engr""","""young oak kim ctr dept""",978383.0,728362.0,24735.0,728979.0,"""lecturer""","""Social Sciences & Communicatio…","""zygmunt pizlo""","""tenure_track"""


In [229]:
duckdb.sql("""
SELECT COUNT(*) AS count, department, role, orig_title, final_title
FROM df
GROUP BY department, role, orig_title, final_title
LIMIT 200
""").pl()

count,department,role,orig_title,final_title
i64,str,str,str,str
862,"""Humanities & Languages""","""lecturer""","""lect-miscellaneous/part time""","""lecturer"""
309,"""Science & Mathematics""","""lecturer""","""adj instr-fy""","""lecturer"""
3166,"""Business, Economics & Finance""","""ladder_faculty""","""prof-ay-b/e/e""","""tenure_track"""
1495,"""Engineering""","""ladder_faculty""","""assoc prof-ay-b/e/e""","""tenure_track"""
…,…,…,…,…
1,"""Social Sciences & Communicatio…","""ladder_faculty""","""asst adj prof-fy""","""tenure_track"""
1,"""Computer Science""","""ladder_faculty""","""assoc prof-ay-1/9-b/e/e""","""tenure_track"""
1,"""Engineering""","""ladder_faculty""","""act assoc prof-ay""","""tenure_track"""
